# Chapitre 8 — Modèles à énergie et ouverture recherche
## Atelier à compléter · M2 IA
PyTorch CPU, Matplotlib. Une densité 2D connue pour juger trois générateurs.


In [ ]:
import math, time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(4)
print('PyTorch :', torch.__version__)


## Atelier 1 · Huit gaussiennes sur un cercle, et deux mesures


In [ ]:
# CELLULE 2 — Une densité 2D connue : huit gaussiennes sur un cercle.
MODES = torch.tensor([[2 * math.cos(2 * math.pi * k / 8), 2 * math.sin(2 * math.pi * k / 8)] for k in range(8)])
SIGMA = .15
def sample_data(n, gen):
    k = torch.randint(0, 8, (n,), generator=gen)
    return MODES[k] + SIGMA * torch.randn(n, 2, generator=gen)

def coverage(samples):
    """Qualité : part des échantillons à moins de 3σ d'un mode. Couverture : modes recevant au moins 2 %."""
    d = torch.cdist(samples, MODES)
    near, k = d.min(1)
    good = near < 3 * SIGMA
    counts = torch.bincount(k[good], minlength=8).float() / len(samples)
    return dict(quality=good.float().mean().item(), modes=int((counts > .02).sum()), counts=counts.tolist())

gen = torch.Generator().manual_seed(0)
X = sample_data(4000, gen)
print('Données :', tuple(X.shape), '; contrôle sur les données elles-mêmes :', coverage(X)['quality'], coverage(X)['modes'])


### Modèle à énergie : maximum de vraisemblance et négatifs par Langevin (≈ 1 min)


In [ ]:
# CELLULE 3 — Un modèle à énergie appris par maximum de vraisemblance, négatifs par Langevin.
class Energy(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

def langevin(E, x, steps, step_size, noise=True, gen=None):
    """x ← x − (ε/2) ∇E(x) + √ε ξ : une marche qui descend l'énergie en restant aléatoire."""
    # TODO : steps pas : gradient de E (autograd), demi-pas de descente, bruit √ε·ξ si noise ; retourner x détaché
    raise NotImplementedError("steps pas : gradient de E (autograd), demi-pas de descente, bruit √ε·ξ si noise ; retourner x détaché")


def train_ebm(steps=3000, seed=0, k_mcmc=30, step_size=.01, alpha=.1):
    torch.manual_seed(seed); gen = torch.Generator().manual_seed(seed)
    E = Energy(); opt = torch.optim.Adam(E.parameters(), lr=1e-3)
    buffer = 6 * torch.rand(8192, 2, generator=gen) - 3          # tampon persistant de négatifs
    history = []
    for step in range(steps):
        pos = sample_data(256, gen)
        idx = torch.randint(0, len(buffer), (256,), generator=gen)
        init = buffer[idx]
        fresh = torch.rand(256, generator=gen) < .05                  # 5 % de redémarrages uniformes
        init[fresh] = 6 * torch.rand(int(fresh.sum()), 2, generator=gen) - 3
        neg = langevin(E, init, k_mcmc, step_size, gen=gen)
        buffer[idx] = neg
        e_pos, e_neg = E(pos), E(neg)
        # Gradient de la log-vraisemblance : abaisser l'énergie des données, relever celle des échantillons du modèle.
        loss = e_pos.mean() - e_neg.mean() + alpha * (e_pos.pow(2).mean() + e_neg.pow(2).mean())
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 100 == 0:
            history.append((step, e_pos.mean().item(), e_neg.mean().item()))
    return E, history

t0 = time.time()
ebm, ebm_history = train_ebm()
gen = torch.Generator().manual_seed(1)
ebm_samples = langevin(ebm, 6 * torch.rand(2000, 2, generator=gen) - 3, 300, .01, gen=gen)
print(f'EBM ({time.time()-t0:.0f} s) :', {k: v for k, v in coverage(ebm_samples).items() if k != 'counts'})


## Atelier 2 · Score matching débruité et Langevin recuit


In [ ]:
# CELLULE 4 — Score matching débruité à plusieurs niveaux de bruit, puis Langevin recuit.
SIGMAS = torch.exp(torch.linspace(math.log(1.), math.log(.01), 10))     # σ₁ > … > σ_L

class Score(nn.Module):
    """s_θ(x, σ) ≈ ∇ₓ log p_σ(x) ; le niveau de bruit est une entrée du réseau."""
    def __init__(self, h=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 2))
    def forward(self, x, sigma):
        return self.net(torch.cat([x, torch.log(sigma)], 1)) / sigma     # paramétrisation usuelle : sortie / σ

def train_dsm(steps=4000, seed=0):
    # TODO : bruiter x à un σ tiré dans SIGMAS, cible −bruit/σ, perte σ²‖s_θ − cible‖² ; retourner (modèle, historique)
    raise NotImplementedError("bruiter x à un σ tiré dans SIGMAS, cible −bruit/σ, perte σ²‖s_θ − cible‖² ; retourner (modèle, historique)")


@torch.no_grad()
def annealed_langevin(S, n=2000, eps=2e-4, T=100, seed=1):
    gen = torch.Generator().manual_seed(seed)
    x = 6 * torch.rand(n, 2, generator=gen) - 3
    for sigma in SIGMAS:
        a = eps * (sigma / SIGMAS[-1]) ** 2                             # pas proportionnel à σ²
        s = torch.full((n, 1), float(sigma))
        for _ in range(T):
            x = x + .5 * a * S(x, s) + torch.sqrt(a) * torch.randn(x.shape, generator=gen)
    return x

t0 = time.time()
score_model, dsm_history = train_dsm()
dsm_samples = annealed_langevin(score_model)
print(f'Score matching ({time.time()-t0:.0f} s) :', {k: v for k, v in coverage(dsm_samples).items() if k != 'counts'})


### Diffusion (DDPM) : prédire le bruit


In [ ]:
# CELLULE 5 — Diffusion (DDPM) : prédire le bruit, c'est estimer le score.
T_STEPS = 200
betas = torch.linspace(1e-4, .04, T_STEPS)
alphas = 1 - betas
abar = torch.cumprod(alphas, 0)

class EpsNet(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 2))
    def forward(self, x, t):
        return self.net(torch.cat([x, (t.float() / T_STEPS)[:, None]], 1))

def train_ddpm(steps=4000, seed=0):
    torch.manual_seed(seed); gen = torch.Generator().manual_seed(seed)
    net = EpsNet(); opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    for step in range(steps):
        x0 = sample_data(512, gen)
        t = torch.randint(0, T_STEPS, (512,), generator=gen)
        eps = torch.randn(x0.shape, generator=gen)
        xt = abar[t].sqrt()[:, None] * x0 + (1 - abar[t]).sqrt()[:, None] * eps
        loss = F.mse_loss(net(xt, t), eps)
        opt.zero_grad(); loss.backward(); opt.step()
    return net

@torch.no_grad()
def ddpm_sample(net, n=2000, seed=1, keep=(199, 150, 100, 50, 0)):
    # TODO : partir de N(0, I), puis pour t = T−1 … 0 : retirer le bruit prédit, diviser par √α_t, ajouter √β_t·ξ si t > 0 ; retourner (x, instantanés)
    raise NotImplementedError("partir de N(0, I), puis pour t = T−1 … 0 : retirer le bruit prédit, diviser par √α_t, ajouter √β_t·ξ si t > 0 ; retourner (x, instantanés)")


t0 = time.time()
ddpm = train_ddpm()
ddpm_samples, ddpm_snapshots = ddpm_sample(ddpm)
print(f'DDPM ({time.time()-t0:.0f} s) :', {k: v for k, v in coverage(ddpm_samples).items() if k != 'counts'})


## Atelier 3 · Contre-expériences
Prédisez qualité, couverture et dispersion avant d’exécuter.


In [ ]:
# CELLULE 6 — Deux contre-expériences : inférer n'est pas échantillonner ; un seul niveau de bruit ne suffit pas.
def spread(samples):
    """Écart-type moyen autour du mode le plus proche (données : σ = 0,15)."""
    d = torch.cdist(samples, MODES); k = d.argmin(1)
    return float((samples - MODES[k]).std(0).mean())

def balance(samples):
    c = torch.tensor(coverage(samples)['counts']); c = c / c.sum()
    return float(-(c * torch.log(c + 1e-12)).sum() / math.log(8))      # 1 = modes équilibrés

gen = torch.Generator().manual_seed(1)
start = 6 * torch.rand(2000, 2, generator=gen) - 3
descent = langevin(ebm, start, 300, .01, noise=False)                   # inférence : argmin E
SIGMAS_SAVE = SIGMAS
SIGMAS = SIGMAS_SAVE[-1:]                                               # un seul niveau : σ = 0,01
single, _ = train_dsm()
single_samples = annealed_langevin(single, T=1000)
SIGMAS = SIGMAS_SAVE

table = {'données': X[:2000], 'EBM, Langevin': ebm_samples, 'EBM, descente sans bruit': descent,
         'score, un seul σ': single_samples, 'score, σ recuit': dsm_samples, 'diffusion (DDPM)': ddpm_samples}
print(f'{"":26s} qualité  modes  équilibre  dispersion')
for name, s in table.items():
    c = coverage(s)
    print(f'{name:26s} {c["quality"]:.3f}    {c["modes"]}      {balance(s):.3f}      {spread(s):.3f}')
